<a href="https://colab.research.google.com/github/penajuanmanuel6-hub/automated-etl-pipeline/blob/main/etl_telemetria_videojuegos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3
import pandas as pd

# 1. Creamos datos de prueba directamente (simulando un archivo crudo con duplicados, nulos y fechas erróneas)
datos_prueba = pd.DataFrame({
    'Jugador_ID': [101, 102, 101, 103, 104],
    'Puntaje': [1500, None, 1500, 3200, 850],
    'Timestamp': ['2026-10-06 10:00:00', '2026-10-06 10:05:00', '2026-10-06 10:00:00', 'Fecha_Invalida', '2026-10-06 10:15:00']
})

print("--- 1. DATOS CRUDOS DE ENTRADA (Con imperfecciones) ---")
display(datos_prueba)

# 2. Transformación y limpieza metódica
def transformar_telemetria(df):
    # Eliminar duplicados exactos
    df = df.drop_duplicates()

    # Estandarizar nombres de columnas a minúsculas sin espacios
    df.columns = [col.strip().lower() for col in df.columns]

    # Manejo de nulos en puntajes (reemplazar por 0)
    df['puntaje'] = df['puntaje'].fillna(0)

    # Convertir marcas de tiempo y eliminar las que tengan errores
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    return df

df_clean = transformar_telemetria(datos_prueba)

print("\n--- 2. DATOS LIMPIOS Y NORMALIZADOS ---")
display(df_clean)

# 3. Carga limpia a base de datos SQLite
conn = sqlite3.connect('telemetria_juego.db')
df_clean.to_sql('sesiones_jugadores', conn, if_exists='replace', index=False)
conn.close()

print("\n--- 3. RESULTADO DE LA CARGA ---")
print("Pipeline ejecutado con éxito. Base de datos SQLite generada y tabla 'sesiones_jugadores' creada localmente.")

--- 1. DATOS CRUDOS DE ENTRADA (Con imperfecciones) ---


,Jugador_ID,Puntaje,Timestamp
0,101,1500.0,2026-10-06 10:00:00
1,102,NaN,2026-10-06 10:05:00
2,101,1500.0,2026-10-06 10:00:00
3,103,3200.0,Fecha_Invalida
4,104,850.0,2026-10-06 10:15:00



--- 2. DATOS LIMPIOS Y NORMALIZADOS ---


/tmp/ipykernel_5263/174102098.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['puntaje'] = df['puntaje'].fillna(0)
/tmp/ipykernel_5263/174102098.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')


,jugador_id,puntaje,timestamp
0,101,1500.0,2026-10-06 10:00:00
1,102,0.0,2026-10-06 10:05:00
4,104,850.0,2026-10-06 10:15:00



--- 3. RESULTADO DE LA CARGA ---
Pipeline ejecutado con éxito. Base de datos SQLite generada y tabla 'sesiones_jugadores' creada localmente.
